# Week 2 — 타임랩스 생성
**환경**: Python 3.10.11 / Windows / VSCode Jupyter

**동작**
- `sleep_frames/날짜/` 폴더에 저장된 이미지를 시간 순으로 이어붙여 mp4 생성
- 정기 촬영 이미지 + 움직임 감지 이미지 모두 포함
- 각 프레임에 타임스탬프 + 자세 라벨 오버레이
- 출력: `./timelapse/YYYYMMDD_timelapse.mp4`

**실행 순서**: 셀을 위에서부터 순서대로 실행하세요 (Shift+Enter)

## 0. 패키지 설치
처음 한 번만 실행하면 됩니다

In [ ]:
%pip install opencv-python numpy matplotlib

## 1. 라이브러리 로드

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import requests
import sys
from pathlib import Path
from datetime import datetime

sys.path.insert(0, "..")     # backend/ → utils 패키지 접근용
sys.path.insert(0, "../..")  # 프로젝트 루트 → config.py 접근용

from utils.db import fetch_posture_by_date

print(f"OpenCV 버전: {cv2.__version__}")
print("[OK] 라이브러리 로드 완료")

## 2. 설정값

In [ ]:
# =============================================
#  설정값 — 필요 시 여기서 수정
# =============================================

# 조회할 날짜 (Supabase posture_log 기준)
# 다른 날짜를 쓰려면 직접 입력: ex) "2025-05-07"
TARGET_DATE = datetime.now().strftime("%Y-%m-%d")

TIMELAPSE_DIR = Path("./timelapse")
TIMELAPSE_DIR.mkdir(parents=True, exist_ok=True)

TIMELAPSE_FPS    = 10    # 재생 속도 (fps)
TIMELAPSE_WIDTH  = 1280  # 출력 해상도 가로
TIMELAPSE_HEIGHT = 720   # 출력 해상도 세로
SHOW_TIMESTAMP   = True  # 프레임에 촬영 시각 표시 여부
SHOW_LABEL       = True  # 프레임 종류(정기/움직임) 표시 여부

OUTPUT_PATH = TIMELAPSE_DIR / f"{TARGET_DATE.replace('-', '')}_timelapse.mp4"

print(f"조회 날짜  : {TARGET_DATE}")
print(f"출력 파일  : {OUTPUT_PATH.resolve()}")
print(f"재생 속도  : {TIMELAPSE_FPS} fps")
print(f"해상도     : {TIMELAPSE_WIDTH} x {TIMELAPSE_HEIGHT}")

## 3. 저장된 이미지 목록 불러오기

In [ ]:
def load_frames_from_supabase(target_date: str) -> list[dict]:
    """
    Supabase posture_log에서 해당 날짜 데이터를 프레임 목록으로 반환
    반환 형식: [{url, timestamp, label}, ...]
    """
    rows = fetch_posture_by_date(target_date)
    return [
        {
            "url":       row["image_path"],    # Storage URL 또는 로컬 경로
            "timestamp": row["timestamp"],
            "label":     row["capture_type"],  # 'regular' or 'motion'
        }
        for row in rows
        if row.get("image_path")
    ]


frames = load_frames_from_supabase(TARGET_DATE)

regular_frames = [f for f in frames if f["label"] == "regular"]
motion_frames  = [f for f in frames if f["label"] == "motion"]

print(f"Supabase에서 불러온 프레임: {len(frames)}장")
print(f"  정기 촬영  : {len(regular_frames)}장")
print(f"  움직임 감지: {len(motion_frames)}장")

if frames:
    start_dt = datetime.fromtimestamp(frames[0]["timestamp"])
    end_dt   = datetime.fromtimestamp(frames[-1]["timestamp"])
    duration = (frames[-1]["timestamp"] - frames[0]["timestamp"]) / 3600
    print(f"\n수면 시작: {start_dt.strftime('%H:%M:%S')}")
    print(f"수면 종료: {end_dt.strftime('%H:%M:%S')}")
    print(f"총 시간  : {duration:.1f}시간")
    timelapse_sec = len(frames) / TIMELAPSE_FPS
    print(f"\n타임랩스 예상 길이: {timelapse_sec:.1f}초 ({timelapse_sec/60:.1f}분)")
else:
    print(f"\n{TARGET_DATE} 날짜의 데이터가 없습니다.")
    print("week2_01 또는 week2_02 노트북을 실행해서 데이터를 먼저 수집하세요.")

## 4. 프레임 오버레이 함수 정의

In [ ]:
LABEL_STYLE = {
    "regular": {"text": "정기 촬영",   "color": (100, 220, 100)},
    "motion":  {"text": "움직임 감지", "color": (80,  80,  255)},
}


def add_overlay(frame: np.ndarray, timestamp: int, label: str,
                frame_idx: int, total: int) -> np.ndarray:
    """프레임에 타임스탬프, 라벨, 진행 바 오버레이"""
    out = frame.copy()
    h, w = out.shape[:2]

    # 상단 반투명 박스
    if SHOW_TIMESTAMP or SHOW_LABEL:
        bar = out.copy()
        cv2.rectangle(bar, (0, 0), (w, 50), (0, 0, 0), -1)
        cv2.addWeighted(bar, 0.5, out, 0.5, 0, out)

    # 촬영 시각
    if SHOW_TIMESTAMP:
        dt_str = datetime.fromtimestamp(timestamp).strftime("%Y-%m-%d  %H:%M:%S")
        cv2.putText(out, dt_str, (12, 32),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)

    # 라벨 (정기/움직임)
    if SHOW_LABEL:
        style = LABEL_STYLE.get(label, {"text": label, "color": (200, 200, 200)})
        cv2.putText(out, style["text"], (w - 200, 32),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, style["color"], 2)

    # 하단 진행 바
    progress_w = int((frame_idx / max(total - 1, 1)) * w)
    cv2.rectangle(out, (0, h - 8), (w, h), (50, 50, 50), -1)
    cv2.rectangle(out, (0, h - 8), (progress_w, h), (80, 200, 120), -1)

    return out


print("[OK] 오버레이 함수 정의 완료")

## 5. 타임랩스 생성

In [ ]:
def read_image(url_or_path: str) -> np.ndarray | None:
    """Storage URL 또는 로컬 경로에서 이미지 읽기"""
    if url_or_path.startswith("http"):
        try:
            resp = requests.get(url_or_path, timeout=10)
            if resp.status_code == 200:
                arr = np.frombuffer(resp.content, np.uint8)
                return cv2.imdecode(arr, cv2.IMREAD_COLOR)
        except Exception as e:
            print(f"  [다운로드 실패] {e}")
        return None
    return cv2.imread(url_or_path)


if not frames:
    print("프레임 없음 — 3번 셀을 먼저 실행하세요")
else:
    writer = cv2.VideoWriter(
        str(OUTPUT_PATH),
        cv2.VideoWriter_fourcc(*"mp4v"),
        TIMELAPSE_FPS,
        (TIMELAPSE_WIDTH, TIMELAPSE_HEIGHT)
    )

    print(f"타임랩스 생성 시작 — 총 {len(frames)}프레임")
    errors = 0

    for i, frame_info in enumerate(frames):
        img = read_image(frame_info["url"])

        if img is None:
            errors += 1
            continue

        img = cv2.resize(img, (TIMELAPSE_WIDTH, TIMELAPSE_HEIGHT))
        img = add_overlay(
            img,
            frame_info["timestamp"],
            frame_info["label"],
            i, len(frames)
        )
        writer.write(img)

        if (i + 1) % 10 == 0 or i == len(frames) - 1:
            pct = (i + 1) / len(frames) * 100
            print(f"  진행중... {i+1}/{len(frames)} ({pct:.0f}%)")

    writer.release()

    file_mb = OUTPUT_PATH.stat().st_size / (1024 * 1024)
    print(f"\n[완료] 타임랩스 생성 성공")
    print(f"  저장 경로 : {OUTPUT_PATH.resolve()}")
    print(f"  파일 크기 : {file_mb:.1f} MB")
    print(f"  총 프레임 : {len(frames) - errors}장")
    print(f"  재생 시간 : {(len(frames) - errors) / TIMELAPSE_FPS:.1f}초")
    if errors:
        print(f"  읽기 실패 : {errors}장 (건너뜀)")

## 6. 타임랩스 썸네일 미리보기
생성된 타임랩스에서 균등 간격으로 8장을 뽑아 노트북 안에서 확인합니다

In [ ]:
if not OUTPUT_PATH.exists():
    print("타임랩스 파일 없음 — 5번 셀을 먼저 실행하세요")
else:
    cap         = cv2.VideoCapture(str(OUTPUT_PATH))
    total_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    thumb_count = min(8, total_frame)
    indices     = np.linspace(0, total_frame - 1, thumb_count, dtype=int)

    thumbs = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            thumbs.append((idx, cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()

    cols = 4
    rows = (len(thumbs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 3))
    axes = axes.flatten()

    for i, (idx, img) in enumerate(thumbs):
        axes[i].imshow(cv2.resize(img, (320, 180)))
        axes[i].set_title(f"프레임 {idx}", fontsize=9)
        axes[i].axis("off")

    for j in range(len(thumbs), len(axes)):
        axes[j].axis("off")

    plt.suptitle(
        f"타임랩스 썸네일 — {total_frame}프레임 / "
        f"{total_frame / TIMELAPSE_FPS:.1f}초",
        fontsize=12
    )
    plt.tight_layout()
    plt.show()

    print(f"재생하려면 아래 경로에서 파일을 직접 열어보세요:")
    print(f"  {OUTPUT_PATH.resolve()}")

## 7. (선택) 테스트용 더미 이미지로 타임랩스 생성
아직 수면 데이터가 없을 때 타임랩스 기능 자체를 먼저 테스트합니다  
웹캠으로 10장 촬영 → 즉시 타임랩스 생성

In [ ]:
import time

DUMMY_DIR  = Path("./sleep_frames/dummy")
DUMMY_DIR.mkdir(parents=True, exist_ok=True)
DUMMY_OUT  = TIMELAPSE_DIR / "dummy_timelapse.mp4"
SHOOT_COUNT = 10   # 촬영 장수
SHOOT_INTERVAL = 0.5  # 촬영 간격 (초)

print(f"{SHOOT_COUNT}장 촬영 후 타임랩스를 바로 생성합니다...")

cap    = cv2.VideoCapture(0)
dummy_frames = []

if not cap.isOpened():
    print("[오류] 웹캠 없음")
else:
    for i in range(SHOOT_COUNT):
        ret, frame = cap.read()
        if ret:
            ts   = int(time.time())
            path = DUMMY_DIR / f"{ts}_regular_color.png"
            cv2.imwrite(str(path), frame)
            dummy_frames.append({"path": path, "timestamp": ts, "label": "regular"})
            print(f"  촬영 {i+1}/{SHOOT_COUNT}")
        time.sleep(SHOOT_INTERVAL)

    cap.release()

    # 타임랩스 생성
    writer = cv2.VideoWriter(
        str(DUMMY_OUT),
        cv2.VideoWriter_fourcc(*"mp4v"),
        TIMELAPSE_FPS,
        (TIMELAPSE_WIDTH, TIMELAPSE_HEIGHT)
    )
    for i, fi in enumerate(dummy_frames):
        img = cv2.imread(str(fi["path"]))
        if img is not None:
            img = cv2.resize(img, (TIMELAPSE_WIDTH, TIMELAPSE_HEIGHT))
            img = add_overlay(img, fi["timestamp"], fi["label"],
                              i, len(dummy_frames))
            writer.write(img)
    writer.release()

    print(f"\n[완료] 더미 타임랩스 생성")
    print(f"  저장 경로: {DUMMY_OUT.resolve()}")